# 00 - Setup, GPU, dan Konfigurasi

**Tujuan:** Menyiapkan lingkungan Colab, menyimpan konfigurasi proyek, dan membuat struktur folder hasil di Drive.

| | |
|---|---|
| **Input** | `mmfood-multitask.zip` di Google Drive |
| **Output** | `artifacts/config.json` dan seluruh struktur folder `mmfood-project/` di Drive |
| **Prasyarat** | Zip proyek sudah di-upload ke `MyDrive` |

> Setiap notebook berdiri sendiri: ia membaca hasil notebook sebelumnya dari Google Drive, jadi sesi Colab boleh diputus/diulang di antara notebook.

## Langkah 0.1 - Siapkan zip di Drive
Upload `mmfood-multitask.zip` ke `MyDrive` (folder paling atas Drive). Lalu aktifkan GPU: `Runtime > Change runtime type > T4 GPU`.

In [5]:
# --- Bootstrap (sama di setiap notebook) ---------------------------------------
# 1) Sambungkan Google Drive (tempat semua hasil disimpan permanen)
import os, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

# 2) Lokasi di Drive: folder ini sekaligus ROOT proyek (zip dan semua hasil ada di sini)
ROOT_ON_DRIVE = "/content/drive/MyDrive/models-train/mmfood-multitask"
ZIP_ON_DRIVE = f"{ROOT_ON_DRIVE}/mmfood-multitask.zip"

# 3) Folder hasil = ROOT di Drive. Harus diset SEBELUM mengimpor config
os.environ["MMFOOD_PROJECT"] = ROOT_ON_DRIVE

# 4) Ekstrak kode + dataset dari zip ke disk lokal Colab (hanya sekali per sesi)
REPO = "/content/mmfood-multitask"
if not os.path.exists(REPO):
    with zipfile.ZipFile(ZIP_ON_DRIVE) as z:
        z.extractall("/content")

# 5) Daftarkan folder src/ agar modul proyek bisa di-import
sys.path.insert(0, f"{REPO}/src")

# 6) Muat path folder (P) dan konfigurasi (CFG) dari src/config.py
from config import get_paths, load_cfg
P = get_paths()       # P.processed, P.checkpoints, P.hf_export, ... (folder dibuat otomatis)
CFG = load_cfg(P)     # nilai bawaan + perubahan dari notebook 00
print("Folder hasil:", P.project)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder hasil: /content/drive/MyDrive/models-train/mmfood-multitask


## Langkah 0.2 - Cek GPU
Training (notebook 04) butuh GPU. Notebook 00-03 tetap bisa jalan tanpa GPU.

In [6]:
import torch

# Cetak status GPU; jika False, ubah runtime ke GPU sebelum notebook 04
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA tersedia: True
GPU: Tesla T4


## Langkah 0.3 - Install library

In [7]:
# Install library yang dibutuhkan (Colab biasanya sudah punya sebagian; -q = senyap)
!pip install -q transformers accelerate huggingface_hub safetensors scikit-learn pillow tqdm matplotlib

## Langkah 0.4 - Tinjau dan ubah konfigurasi
Semua pengaturan ada di satu dict `CFG`. Ubah nilai di sel di bawah bila perlu, lalu sel berikutnya menyimpannya ke Drive. Notebook lain akan membaca file yang sama.

Panduan cepat:
- **Uji cepat (10-15 menit):** `n_attempt=8000`, `epochs=2`
- **GPU kehabisan memori:** turunkan `batch_size` ke 32 atau 16
- **`hf_username`:** isi nanti sebelum notebook 06/07

In [8]:
import json

# Contoh mengubah nilai (hapus tanda # untuk memakai):
# CFG["n_attempt"] = 8000
# CFG["epochs"] = 2
# CFG["batch_size"] = 32
# CFG["hf_username"] = "usernamekamu"

print(json.dumps(CFG, indent=2))

{
  "backbone_name": "facebook/convnext-tiny-224",
  "min_food_prob": 0.5,
  "top_k_dish": 50,
  "min_per_cooking": 150,
  "n_attempt": 36000,
  "cache_size": 256,
  "download_workers": 32,
  "img_size": 224,
  "batch_size": 64,
  "epochs": 6,
  "lr_backbone": 5e-05,
  "lr_heads": 0.001,
  "weight_decay": 0.05,
  "loss_weights": [
    1.0,
    1.0,
    0.5,
    1.0
  ],
  "num_workers": 2,
  "seed": 42,
  "hf_username": "GANTI_USERNAME_HF",
  "model_name": "convnext-tiny-mmfood-multitask",
  "hf_private": true
}


In [9]:
from config import save_cfg

# Simpan konfigurasi ke Drive: artifacts/config.json (dibaca semua notebook)
save_cfg(P, CFG)
print("Tersimpan:", f"{P.artifacts}/config.json")

Tersimpan: /content/drive/MyDrive/models-train/mmfood-multitask/artifacts/config.json


## Langkah 0.5 - Lihat struktur folder hasil

In [10]:
# Tampilkan pohon folder proyek di Drive (folder kosong normal pada tahap ini)
for root, dirs, files in os.walk(P.project):
    depth = root[len(P.project):].count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    for f in files:
        print("  " * (depth + 1) + f)

mmfood-multitask/
  mmfood-multitask.zip
  README.md
  requirements.txt
  notebooks/
    00_setup_config.ipynb
    01_eda_cleaning.ipynb
    02_download_images.ipynb
    03_split_labels.ipynb
    04_train_model.ipynb
    05_evaluate.ipynb
    06_save_export_model.ipynb
    07_push_to_huggingface.ipynb
    08_package_results.ipynb
  data/
    raw/
      MM-Food-100K.csv
    processed/
    image_cache/
  docs/
    DATASET_NOTES.md
    HF_NAMING_AND_PUSH_GUIDE.md
    MODEL_CARD_TEMPLATE.md
    TROUBLESHOOTING.md
  src/
    config.py
    data_utils.py
    dataset.py
    images.py
    modeling_mmfood.py
    train_utils.py
  artifacts/
    config.json
  checkpoints/
  reports/
    metrics/
    figures/
  models/
    raw/
    hf_export/
  releases/
